# 9b. Additions: Iterables, and Changing a List in a Loop

Additions to [Chapter 9: Lists](https://y-rosenthal.github.io/yrThinkPython/chap09.html): a review of the
**iterables** that a `for` loop can go through, and a rule that you must never break: **do not change a list
in the body of a loop that goes through that list**. A list is the first iterable in the book that a program
can change, and the page shows what goes wrong when a loop changes it. Read this page after the chapter.
The practice questions at the end have answers: try each question first, then open its **Answer**.

[Run this page on Colab](https://colab.research.google.com/github/y-rosenthal/yrThinkPython/blob/v3/yr/chap09_additions_iterables.ipynb)

Run the setup cell below first. In Colab, it downloads the file `tiny_book.txt` (a small "book" of 4 lines
that this page reads), and `thinkpython.py`, which lets this page show the errors of some examples without
stopping the page.

In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://y-rosenthal.github.io/yrThinkPython/tiny_book.txt');

import thinkpython

## Review: what is an iterable?

An **iterable** is a value that a `for` loop can go through, one item at a time:

```python
for VARIABLE in ITERABLE:
    BODY
```

These are the iterables of the book up to Chapter 9:

| Iterable | What the loop variable gets | Where the book teaches it |
|---|---|---|
| a range, such as `range(3)` | the numbers of the range | Chapter 3, and "3b. Additions: The Three Forms of range" |
| a string, such as `'cat'` | each character of the string | Chapter 7 |
| a file object, from `open(PATH)` | each line of the file, as a string with a newline at the end | Chapter 7, and "7c. Additions: File Paths and for Loops" |
| `reversed(ITERABLE)` | the items of another iterable, in the opposite order | "7c. Additions: File Paths and for Loops" |
| a list, such as `[10, 20, 30]` | each item of the list | Chapter 9 |

Later chapters add more: a dictionary (Chapter 10) gives its keys, and a tuple (Chapter 11) gives its items.
A number is not an iterable: `for x in 5:` raises a `TypeError`.

**A range** gives numbers:

In [ ]:
for number in range(1, 4):
    print(number)

**A string** gives its characters:

In [ ]:
for letter in 'cat':
    print(letter)

**A file object** gives the lines of the file. This page uses the small file `tiny_book.txt`, which has 4
lines. In Colab, the setup cell downloaded it. To run the code on your own computer, download the file:
[tiny_book.txt](https://y-rosenthal.github.io/yrThinkPython/tiny_book.txt) (right-click the link and choose to save it), and give `open` its path
("7c. Additions: File Paths and for Loops"). `strip` removes the newline at the end of each line:

In [ ]:
for line in open('tiny_book.txt'):
    print(line.strip())

**A list** gives its items:

In [ ]:
for word in ['apple', 'banana', 'cherry']:
    print(word, len(word))

## Changing a list in a loop over the list

Chapter 9 shows that lists are **mutable**: a program can change a list after it makes it. The method
`append` adds an item at the end of a list, and the method `remove` takes an item out. This section shows
what happens when the body of a loop changes the list that the loop goes through.

**The task:** write a function called `remove_evens` that takes a list of numbers, removes the even numbers,
and returns the list with the numbers that are left. Here is a first try. **It is wrong:**

In [ ]:
def remove_evens(numbers):
    for n in numbers:
        if n % 2 == 0:
            numbers.remove(n)
    return numbers

remove_evens([1, 2, 3, 4, 5, 6])

The result is right: the even numbers 2, 4 and 6 are gone. But the function has a bug. Try it with other
lists:

In [ ]:
def remove_evens(numbers):
    for n in numbers:
        if n % 2 == 0:
            numbers.remove(n)
    return numbers

print(remove_evens([2, 4, 6, 8]))
print(remove_evens([1, 2, 4, 5]))

The results should be `[]` and `[1, 5]`. The function left `4` and `8` in the first list, and `4` in the
second. There is no error message: the function just returns a wrong list. A bug like this is hard to find,
because a test with one list (the first one above) can pass while the code is wrong.

To see what happens, display each value that the loop gives to `n`:

In [ ]:
numbers = [2, 4, 6, 8]
for n in numbers:
    print('n is', n, 'and the list is', numbers)
    if n % 2 == 0:
        numbers.remove(n)
print('After the loop:', numbers)

The loop never gives `4` or `8` to `n`. Step by step:

| Iteration | Position that the loop reads | `n` | The list after `remove` |
|---|---|---|---|
| 1 | 0 | `2` | `[4, 6, 8]`: every item moved one position to the left |
| 2 | 1 | `6` (not `4`: `4` is now at position 0) | `[4, 8]` |
| 3 | 2 | the list has no position 2, so the loop ends | `[4, 8]` |

Each `remove` moves the items after it one position to the left, so the loop skips the item right after
each item that it removes. With `[1, 2, 3, 4, 5, 6]`, no two even numbers are next to each other, so the
skipped items were odd numbers, and the result was right by chance.

**Adding items is worse.** In this loop, each iteration adds an item at the end of the list. The loop never
gets to the end of the list, so **the loop never ends**:

```python
numbers = [1, 2, 3]
for n in numbers:
    numbers.append(n)    # never do this: an infinite loop
```

This code is text, not a code cell, so it cannot run. If you run such a loop, stop it: in Colab, click
**Runtime**, then **Interrupt execution**; in Thonny, click the red **Stop** button.

## Why the bug happens: a list is an iterable that can change

A `for` loop over a list does **not** make a copy of the list when it starts. It goes through the list itself,
by position:

1. The loop starts at position 0.
2. If the position is less than the length that the list has **now**, the loop assigns the item at that
   position to the loop variable and runs the body. Otherwise the loop ends.
3. The loop adds 1 to the position, and goes back to step 2.

So when the body changes the list, the loop goes on through the **changed** list:

- When the body removes an item, the items after it move one position to the left, and the loop skips one
  item.
- When the body adds an item, the list gets longer, and the loop gets to the new item too. A loop that adds an
  item in each iteration never ends.
- When the body assigns a new value to an item that the loop has not got to yet (for example,
  `numbers[3] = 0`), the loop gets the new value, not the old one.

**The rule:** in the body of a `for` loop that goes through a list, **never change that list**: do not add,
remove or replace its items. Code that does this is **very bad code**. It can skip items without an error
message, or never end, and a reader cannot see what it does without following each step. Never write such
code.

The function also has a second problem: it changes the list that the caller gave it, because the parameter
and the caller's variable refer to the same list ("List arguments" in Chapter 9).

**The right way:** make a **new** list, and append to it the items to keep. The loop goes through one list,
and the body changes another list:

In [ ]:
def remove_evens(numbers):
    result = []
    for n in numbers:
        if n % 2 != 0:
            result.append(n)
    return result

print(remove_evens([1, 2, 3, 4, 5, 6]))
print(remove_evens([2, 4, 6, 8]))
print(remove_evens([1, 2, 4, 5]))

All three results are right, and the caller's lists do not change.

## An immutable iterable: a string

Strings are **immutable** ("Strings are immutable" in Chapter 8): no method or operator can change a string.
An expression such as `word + '!'` or `word.upper()` makes a **new** string. So the body of a loop over a
string cannot change the string that the loop goes through. Even this loop, which seems to make its string
longer, runs exactly 3 times:

In [ ]:
word = 'cat'
for letter in word:
    word = word + letter
    print(letter, word)
print('After the loop:', word)

At the start, the loop gets the string `'cat'` that `word` refers to, and it goes through that string. The
assignment `word = word + letter` makes a new string and makes the **variable** `word` refer to it. The string
`'cat'` does not change, so the loop goes through its 3 characters and ends. A variable and the value that it
refers to are different things ("Objects and values" in Chapter 9).

The same is true for the other immutable iterables: a range, and a tuple (Chapter 11).

**The difference is between changing a value and assigning a new value to a variable.** A list can show both:

In [ ]:
numbers = [1, 2, 3]
for n in numbers:
    numbers = numbers + [n * 10]
print(numbers)

This loop runs 3 times and ends. The `+` operator makes a **new** list, and the assignment makes the variable
`numbers` refer to it. The list that the loop goes through, `[1, 2, 3]`, does not change. `numbers.append(n * 10)`
would change that list itself, and the loop would never end.

So with an immutable iterable, the body cannot change what the loop goes through. Code like the two loops
above is still bad code: a reader must know the difference between changing a value and assigning a new value
to a variable, to see how many times the loop runs. Use a different variable for the result (for example,
`longer = longer + letter`), and keep the loop's iterable as it is.

## Practice questions

### Question 1 (easy): which are iterables?

Which of these values can follow `in` in the header of a `for` loop?

`range(5)`, `'hello'`, `42`, `[3, 1, 2]`, `open('tiny_book.txt')`, `3.5`, `reversed('abc')`

<details>
<summary>Answer</summary>

`range(5)`, `'hello'`, `[3, 1, 2]`, `open('tiny_book.txt')` and `reversed('abc')` are iterables. `42` and `3.5`
are numbers, not iterables: `for x in 42:` raises `TypeError: 'int' object is not iterable`.

</details>

### Question 2 (easy): what is displayed?

Predict what this code displays. Then run the cell to check (on the website, click **Show code cell output**).

In [ ]:
count = 0
for line in open('tiny_book.txt'):
    if 'loop' in line.lower():
        count = count + 1
print(count)

<details>
<summary>Answer</summary>

```
2
```

The loop gets the 4 lines of `tiny_book.txt`. Two of them have the text `loop`: the first line (`Loops`, made
lowercase by `lower`) and the second line (`A for loop goes ...`).

</details>

### Question 3 (medium): what is displayed?

This is the wrong `remove_evens` from the page. Predict what this code displays. Then run the cell to check
(on the website, click **Show code cell output**).

In [ ]:
def remove_evens(numbers):
    for n in numbers:
        if n % 2 == 0:
            numbers.remove(n)
    return numbers

print(remove_evens([3, 6, 8, 9, 10, 12]))

<details>
<summary>Answer</summary>

```
[3, 8, 9, 12]
```

The loop reads position 1 (`6`) and removes it: `8` moves to position 1, and the loop skips it. Then it reads
position 2 (`9`), and position 3 (`10`), which it removes: `12` moves to position 3, and the loop ends because
the list has no position 4. `8` and `12` are left in the list.

</details>

### Question 4 (medium): what is displayed?

Predict what this code displays. Then run the cell to check (on the website, click **Show code cell output**).

In [ ]:
text = 'ab'
for ch in text:
    text = text + ch.upper()
print(text)

<details>
<summary>Answer</summary>

```
abAB
```

The loop goes through the string `'ab'`, which cannot change, so it runs 2 times. Each assignment makes a new
string and assigns it to `text`: `'abA'`, then `'abAB'`.

</details>

### Question 5 (medium): how many times?

How many times does each loop run? Explain the difference.

```python
first = [5, 6]
for item in first:
    first = first + [item]
```

```python
second = [5, 6]
for item in second:
    second.append(item)
```

<details>
<summary>Answer</summary>

The first loop runs **2 times**. `first + [item]` makes a new list, and the assignment makes `first` refer to
it. The list that the loop goes through, `[5, 6]`, does not change. After the loop, `first` is
`[5, 6, 5, 6]`.

The second loop **never ends**. `append` changes the list that the loop goes through: each iteration adds an
item, so the loop never gets to the end of the list. Both loops are bad code, and the second one is a bug.

</details>

### Question 6 (hard): find the bug

This function is supposed to remove the words that are shorter than `length` from a list of words, and to
return the list with the words that are left. Find the bug, and write a correct version that does not change
the caller's list.

```python
def remove_short_words(words, length):
    for word in words:
        if len(word) < length:
            words.remove(word)
    return words
```

For example, `remove_short_words(['a', 'an', 'the', 'apple'], 3)` should return `['the', 'apple']`.

In [ ]:
# Your code here

<details>
<summary>Answer</summary>

The body removes items from the list that the loop goes through. Each `remove` moves the next word one
position to the left, so the loop skips it. With `['a', 'an', 'the', 'apple']`, the loop removes `'a'`, skips
`'an'`, and returns `['an', 'the', 'apple']`.

A correct version makes a new list:

```python
def remove_short_words(words, length):
    result = []
    for word in words:
        if len(word) >= length:
            result.append(word)
    return result
```

`remove_short_words(['a', 'an', 'the', 'apple'], 3)` returns `['the', 'apple']`, and the caller's list does
not change.

</details>

### Question 7 (hard): write a function

Write a function called `remove_all` that takes a list and a value, and returns a **new** list with all the
items of the list except the ones equal to the value. The function must not change the list that it gets.

For example, `remove_all([1, 2, 1, 1, 3], 1)` returns `[2, 3]`, and `remove_all(['a', 'b'], 'c')` returns
`['a', 'b']`.

Start from this header (replace `...` with the body):

```python
def remove_all(items, value):
    ...
```

In [ ]:
# Your code here

<details>
<summary>Answer</summary>

```python
def remove_all(items, value):
    result = []
    for item in items:
        if item != value:
            result.append(item)
    return result
```

The loop goes through `items`, and the body appends to `result`, a different list. So the loop gets every
item, also the items right after a removed one, and `items` does not change.

</details>

## Credits

*Additions by Prof. Y. Rosenthal, for* [Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html) *by Allen B. Downey.*
*Text license:* [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)